In [7]:
!pip install datasets tensorflow gradio scikit-learn

In [12]:
import pandas as pd
from datasets import load_dataset

try:
    # Using 'GonzaloA/fake_news' as a stable and popular alternative on Hugging Face Hub
    dataset_hf = load_dataset("GonzaloA/fake_news")

    # Combine train/test splits if needed or just use train for now
    data = dataset_hf['train'].to_pandas()

    # Shuffle the data
    data = data.sample(frac=1, random_state=42).reset_index(drop=True)

    # Take a subset for speed if necessary
    data = data.head(10000)

    print(f"Total data size: {len(data)}")
    # Display columns to see the structure (usually 'text', 'title', 'label')
    print(f"Columns: {data.columns.tolist()}")
    display(data.head())

except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")
    print("Please verify your internet connection or Hugging Face Hub status.")

README.md:   0%|          | 0.00/6.73k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Total data size: 10000
Columns: ['Unnamed: 0', 'title', 'text', 'label']


,Unnamed: 0,title,text,label
0,2579,TUCKER CARLSON EXPOSES REFUGEE CONTRACTOR Who ...,WORLD RELIEF received over $43 million dollars...,0
1,13868,Romania negotiating to buy U.S. rocket systems...,WASHINGTON (Reuters) - The U.S. State Departme...,1
2,13020,Judge Who Barred A Mom From Seeing Her Baby F...,With all the recent talk of sentencing reform ...,0
3,3070,Pro-Kurdish opposition leader's trial opens in...,ANKARA (Reuters) - The jailed leader of Turkey...,1
4,4528,White Supremacists Robocall For Trump Ahead O...,A Super PAC representing white nationalists is...,0


In [13]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(data["text"], data["label"], test_size=0.2)

# إعداد التوكنيزر
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

# تحويل النصوص إلى تسلسلات أرقام
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# حفظ التوكنيزر لاستخدامه لاحقاً في Hugging Face
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Preprocessing Complete!")

✅ Preprocessing Complete!


In [14]:
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Embedding(max_words, 128, input_length=max_len),
    layers.SpatialDropout1D(0.2),
    layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# تدريب الموديل (3 Epochs فقط للتجربة)
history = model.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_data=(X_test_pad, y_test))

# حفظ الموديل
model.save("fake_news_model.h5")
print("✅ Model Saved as fake_news_model.h5")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 78s 535ms/step - accuracy: 0.8985 - loss: 0.2794 - val_accuracy: 0.9585 - val_loss: 0.1119
Epoch 2/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 67s 540ms/step - accuracy: 0.9697 - loss: 0.0884 - val_accuracy: 0.9645 - val_loss: 0.0984
Epoch 3/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 65s 521ms/step - accuracy: 0.9843 - loss: 0.0514 - val_accuracy: 0.9495 - val_loss: 0.1203


✅ Model Saved as fake_news_model.h5


In [15]:
!pip install gradio
import gradio as gr

def predict_news(text):
    # تجهيز النص المدخل بنفس طريقة التدريب
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    # التنبؤ
    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        return f"✅ REAL NEWS (Confidence: {prediction:.2%})"
    else:
        return f"❌ FAKE NEWS (Confidence: {1-prediction:.2%})"

# تشغيل الواجهة
demo = gr.Interface(fn=predict_news, inputs="text", outputs="text", title="Fake News Detector")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec6d14662a2a8db967.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
from google.colab import files

# تحميل الموديل
files.download("fake_news_model.h5")

# تحميل التوكنيزر
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>